# WavMamba — WiFi-CSI HAR on UT-HAR & NTU-Fi

Public code accompanying the paper. Trains **WavMamba** — a multi-branch
(one branch per Haar DWT subband) CNN + bidirectional-Mamba model with adaptive
late fusion — for WiFi-CSI human activity recognition on **UT-HAR** and **NTU-Fi**.

Architecture (fixed): Haar `{HL, LH}` branches, attentive statistics pooling,
no stem GroupNorm, per-channel gate fusion.

Normalization is controlled by two orthogonal flags:
- `PRENORM` — pre-norm on raw (before DWT): `sensefi` (UT-HAR min-max / NTU-Fi (x-42.32)/4.98) | `none` (no raw pre-normalization).
- `Z_GRAN`  — z-norm after DWT (always on): `perpos` (per-position `(C,T2,F2)`) | `pcb` (per-channel-bin `(C,F2)`, time collapsed).

Four combinations (`sensefi+perpos`, `none+pcb`, `sensefi+pcb`, `none+perpos`),
each writing its own bench dir, so runs never overwrite.

**Steps:** set `REPO_URL` in the clone cell, install the Mamba kernels + deps,
set `PRENORM`/`Z_GRAN` in the config cell, then run all cells.


In [ ]:
# Cell 2 — Clone / update the public code from GitHub
import sys, subprocess
from pathlib import Path

# EDIT before public release: point REPO_URL at the final public repository.
# Optionally pin REPO_REF to the paper release tag/commit for exact reproduction.
REPO_URL  = 'https://github.com/<owner>/wavmamba.git'
REPO_REF  = None          # e.g. 'v1.0' or a commit SHA; None = default branch HEAD
CODE_PATH = Path('/kaggle/working/wavmamba')

if not CODE_PATH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE_PATH)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)
if REPO_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF],
                   cwd=str(CODE_PATH), check=True)
    subprocess.run(['git', 'checkout', REPO_REF], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code at {CODE_PATH}  (ref={REPO_REF or "HEAD"})')


In [ ]:
# Cell 2b — Install dependencies (run once per session, before any heavy import)
#
# mamba-ssm / causal-conv1d ship prebuilt CUDA wheels that must match the torch
# C++ ABI, so install them FIRST with --no-deps (so pip does not re-resolve or
# replace an ABI-compatible torch), then the regular requirements.
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args], check=True)

# Kaggle images already ship a CUDA torch; skip the explicit torch pin there.
# Uncomment to force a specific torch build locally:
# _pip('torch==2.7.0')
_pip('mamba-ssm', '--no-build-isolation', '--no-deps')
_pip('causal-conv1d', '--no-deps')
_pip('-r', str(CODE_PATH / 'requirements.txt'))

from train import run
from config import default_cfg
print('Install + import OK : train.run')


In [ ]:
# Cell 3 — Configuration
from pathlib import Path

PRENORM   = 'none'        # 'sensefi' = UT-HAR min-max / NTU-Fi (x-42.32)/4.98 | 'none' = no raw pre-normalization
Z_GRAN    = 'pcb'         # 'perpos' = z per-position (C,T2,F2) | 'pcb' = z per-channel-bin (C,F2), time collapsed
MERGE_VAL = True          # ONLY UT-HAR: True = merge val into test | False = test=X_test only
DATASETS  = ['uthar', 'ntufi']
MODES     = ['raw']       # ['raw'] | ['proc'] | ['raw','proc']
SEEDS     = [0, 4, 8, 17, 42]
OUT_ROOT  = '/kaggle/working'

DIRMAP   = {'uthar': 'UT_HAR', 'ntufi': 'NTU-Fi_HAR'}
_MARKER  = {'uthar': 'X_train.csv', 'ntufi': 'train_amp'}
assert PRENORM in ('none', 'sensefi'), f'bad PRENORM {PRENORM!r}'
assert Z_GRAN in ('perpos', 'pcb'),    f'bad Z_GRAN {Z_GRAN!r}'
RUN_TAG  = f'{PRENORM}_{Z_GRAN}' + ('_mv' if MERGE_VAL else '')
build_py = CODE_PATH / 'build_dataset.py'

def resolve_mount(ds):
    base = Path('/kaggle/input')
    for c in (sorted(base.iterdir()) if base.is_dir() else []):
        if next(c.rglob(_MARKER[ds]), None) is not None:
            return str(c)
    raise FileNotFoundError(f'No /kaggle/input/* containing {_MARKER[ds]} for {ds}')

print(f'PRENORM={PRENORM}  Z_GRAN={Z_GRAN}  MERGE_VAL={MERGE_VAL}  RUN_TAG={RUN_TAG}')
print(f'DATASETS={DATASETS}  MODES={MODES}  SEEDS={SEEDS}')
for ds in DATASETS:
    try:    print(f'  {ds:6s} mount: {resolve_mount(ds)}')
    except Exception as e: print(f'  {ds:6s} !! {e}')


In [ ]:
# Cell 4 — Smoke: WavMamba builds + forwards on GPU (fail-fast before build/train)
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
from model import WavMamba
# UT-HAR dims: n_links=1, n_antennas=3, f2=15 -> packed C = 2*3 = 6
_m = WavMamba(num_classes=7, n_links=1, n_antennas=3, f2=15).to(dev)
with torch.no_grad():
    _o = _m(torch.randn(2, 6, 125, 15, device=dev))   # T2=125 (250//2)
assert _o.shape == (2, 7), f'bad output {tuple(_o.shape)}'
print(f'SMOKE OK ({dev}) — WavMamba forward -> {tuple(_o.shape)}, '
      f'{sum(p.numel() for p in _m.parameters()):,} params')
del _m, _o
if dev == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Cell 5 — Sweep: build + train each dataset (separate output dir per run)
import subprocess, sys, time, json, gc, torch
from config import default_cfg

NL = chr(10)

def model_kwargs(meta):
    # Fixed architecture; only dataset dims are passed in.
    return {'n_links': 1, 'n_antennas': meta['n_per_sub'], 'f2': meta['F2']}

def run_one(ds, md_mode):
    raw     = resolve_mount(ds)
    md_sub  = f'{md_mode}_{PRENORM}_{Z_GRAN}'           # match build script bench dir
    bench   = Path(OUT_ROOT) / DIRMAP[ds] / 'bench' / md_sub
    out     = Path(f'{OUT_ROOT}/outputs/wavmamba_{ds}_{md_mode}_{RUN_TAG}')
    cmd = [sys.executable, str(build_py),
           '--dataset', ds, '--mode', md_mode,
           '--raw-root', raw, '--out-root', OUT_ROOT,
           '--prenorm', PRENORM, '--z-gran', Z_GRAN]
    if MERGE_VAL: cmd.append('--merge-val')
    subprocess.run(cmd, check=True)
    meta = json.load(open(bench / 'stats.json'))['meta']
    run(bench_dir=bench, output_dir=out, train_cfg=default_cfg(seeds=SEEDS),
        num_workers=4, model_kwargs=model_kwargs(meta),
        num_classes=meta['classes'], class_names=meta['class_names'],
        dataset_name=ds, split_desc=meta['split'])

results = {}
for ds in DATASETS:
    for md_mode in MODES:
        t0 = time.time()
        banner = NL + '#' * 64 + NL + '#  wavmamba / ' + ds + ' / ' + md_mode + ' / ' + RUN_TAG + NL + '#' * 64
        print(banner)
        try:
            run_one(ds, md_mode); results[f'{ds}/{md_mode}'] = 'OK'
        except Exception as e:
            results[f'{ds}/{md_mode}'] = f'FAILED: {type(e).__name__}: {e}'; print('!!', e)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"== {ds}/{md_mode}: {results[f'{ds}/{md_mode}']}  ({(time.time()-t0)/60:.1f} min)")
print(NL + '=== SWEEP SUMMARY ===')
for k, v in results.items():
    print(f'  {k:14s}: {v}')
